# PyTorch Tutorial

Paul J. Atzberger<br> 
<https://web.atzberger.org/>

# Tensors

A **tensor** is the central data structure in PyTorch. It is a multi-dimensional array, similar to a NumPy array but with GPU support and the ability to track gradients for automatic differentiation.

In this notebook you will learn:
- Many ways to construct tensors
- How to inspect tensor attributes (shape, dtype, device)
- Basic arithmetic and linear-algebra operations
- Indexing, slicing, and reshaping
- How tensors relate to NumPy arrays

In [ ]:
import torch

print("PyTorch version:", torch.__version__)

---
## Creating Tensors from Python Data

`torch.tensor()` converts any Python list (or nested list) into a tensor.
PyTorch infers the data type from the values.

In [ ]:
# 1-D tensor (a vector)
v = torch.tensor([1.0, 2.0, 3.0, 4.0])
print("vector :", v)

# 2-D tensor (a matrix)
M = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
print("matrix :\n", M)

# 3-D tensor (e.g. a batch of images with one channel)
T = torch.tensor([[[1, 2], [3, 4]],
                  [[5, 6], [7, 8]]])
print("3-D tensor shape:", T.shape)

---
## Creating PyTorch Tensors from Scratch

These factory functions let you build tensors of a given shape without supplying
individual values.

In [ ]:
# All zeros / all ones
print(torch.zeros(2, 3))      # 2 rows, 3 columns
print(torch.ones(3, 4))

# Fill with a constant
print(torch.full((2, 3), fill_value=7.0))

# Identity matrix
print(torch.eye(4))           # 4x4 identity

# Uninitialised (fast -values are whatever was in memory)
print(torch.empty(2, 3))

### Random tensors

Always seed the random number generator when you need reproducible results.

In [ ]:
torch.manual_seed(42)

# Uniform in [0, 1)
print("rand  :", torch.rand(2, 3))

# Standard normal (mean=0, std=1)
print("randn :", torch.randn(2, 3))

# Random integers in [low, high)
print("randint:", torch.randint(low=0, high=10, size=(2, 4)))

### Sequential tensors

`arange` and `linspace` are the PyTorch equivalents of `range` and `np.linspace`.

In [ ]:
# arange(start, stop, step)  -stop is exclusive
print(torch.arange(0, 10, 2))          # [0, 2, 4, 6, 8]

# linspace(start, stop, n)  -stop is INCLUSIVE
print(torch.linspace(0, 1, 5))         # [0.00, 0.25, 0.50, 0.75, 1.00]
print(torch.linspace(-3.14, 3.14, 7))  # 7 evenly-spaced points

---
## Tensor Attributes

Every tensor carries three key pieces of metadata:

In [ ]:
x = torch.randn(3, 4, 5)

print("shape  :", x.shape)    # torch.Size([3, 4, 5])
print("ndim   :", x.ndim)     # number of dimensions = 3
print("dtype  :", x.dtype)    # element data type
print("device :", x.device)   # 'cpu' or 'cuda:0'
print("numel  :", x.numel())  # total number of elements = 3*4*5 = 60

---
## Data Types (dtype)

PyTorch supports many numeric types. The default floating-point type is
`torch.float32` (32-bit float). Neural networks almost always use `float32`
because it strikes the right balance between precision and speed.

In [ ]:
a_float32 = torch.tensor([1.0, 2.0, 3.0])                    # default: float32
a_float64 = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
a_int32   = torch.tensor([1, 2, 3], dtype=torch.int32)
a_bool    = torch.tensor([True, False, True])

print(a_float32, a_float32.dtype)
print(a_float64, a_float64.dtype)
print(a_int32,   a_int32.dtype)
print(a_bool,    a_bool.dtype)

# Cast between types with .to() or type-specific shortcuts
print(a_float64.to(torch.float32))
print(a_int32.float())   # shorthand for .to(torch.float32)

---
## Basic Arithmetic

Standard Python operators work element-wise on tensors.  
Every operator has a functional equivalent in `torch.*`.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("add      :", a + b)           # or torch.add(a, b)
print("sub      :", a - b)
print("mul      :", a * b)           # element-wise
print("div      :", a / b)
print("power    :", a ** 2)
print("dot prod :", torch.dot(a, b)) # 1-D dot product
print("sum      :", a.sum())
print("mean     :", a.mean())
print("max      :", a.max())

In [ ]:
# Matrix multiplication
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0],
                  [7.0, 8.0]])

print("A @ B =\n", A @ B)            # @ operator -matrix multiply
print("torch.mm =\n", torch.mm(A, B)) # equivalent function

In [ ]:
# In-place operations -modify the tensor in memory (trailing underscore convention)
x = torch.ones(3)
print("before:", x)
x.add_(10)    # adds 10 in-place
print("after :", x)

# Note: in-place ops on tensors that require gradients will raise an error.
# Prefer out-of-place (x = x + 10) inside training loops.

---
## Indexing and Slicing

Indexing works the same as NumPy.  This includes negative indices, slices, and
advanced (boolean / integer-array) indexing.

In [ ]:
M = torch.arange(12).reshape(3, 4).float()
print("M =\n", M)

print("\nRow 0          :", M[0])          # first row
print("Col 2          :", M[:, 2])         # third column
print("Rows 1-2       :\n", M[1:])         # slicing rows
print("Element [1, 3] :", M[1, 3])        # single element
print("Sub-block      :\n", M[0:2, 1:3])  # 2x2 sub-block

In [ ]:
# Boolean masking -select elements that satisfy a condition
mask = M > 5
print("mask =\n", mask)
print("M[M > 5] =", M[mask])  # returns a 1-D tensor of matching values

---
## Reshaping

Reshaping changes the *view* of the data without copying it.  
The total number of elements must stay the same.

In [ ]:
x = torch.arange(24)
print("original shape:", x.shape)     # [24]

# view / reshape -use -1 to let PyTorch infer one dimension
a = x.view(4, 6)
b = x.reshape(2, 3, 4)
c = x.reshape(6, -1)   # -1 => 24/6 = 4, so shape is [6, 4]

print("view(4,6)    :", a.shape)
print("reshape(2,3,4):", b.shape)
print("reshape(6,-1) :", c.shape)

In [ ]:
# squeeze / unsqueeze -remove or add size-1 dimensions
t = torch.randn(1, 4, 1)
print("original   :", t.shape)            # [1, 4, 1]
print("squeezed   :", t.squeeze().shape)  # [4]    -removes all size-1 dims
print("unsqueeze 0:", t.squeeze().unsqueeze(0).shape)  # [1, 4]

# flatten -collapses all dimensions into one
img = torch.randn(3, 32, 32)  # 3-channel 32x32 image
print("flatten    :", img.flatten().shape)  # [3072]

# permute -reorder dimensions (like transpose for N-D)
hwc = torch.randn(32, 32, 3)   # height x width x channels
chw = hwc.permute(2, 0, 1)     # channels x height x width
print("permute    :", chw.shape)

---
## Interoperability with NumPy

Tensors and NumPy arrays share memory when on a CPU. Codes and APIs converting between them is essentially free (no copy). Modify one and the other changes too.

In [ ]:
import numpy as np

# Tensor → NumPy
t = torch.tensor([1.0, 2.0, 3.0])
arr = t.numpy()
print("numpy array:", arr, type(arr))

# NumPy → Tensor
arr2 = np.array([4.0, 5.0, 6.0])
t2 = torch.from_numpy(arr2)
print("tensor     :", t2, t2.dtype)

# Shared memory: modifying the numpy array changes the tensor
arr[0] = 99
print("t after arr[0]=99:", t)   # t[0] is now 99.0!

---
## Devices: CPU and GPU

PyTorch can run on CPU or on an NVIDIA GPU (CUDA). The device is a property of each tensor. 

**Note:** It is required during operations that both tensors be on the same device.

In [ ]:
# Check whether a GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Create a tensor directly on the target device
x = torch.randn(3, 3, device=device)
print("x.device:", x.device)

# Move an existing tensor with .to()
y = torch.ones(3, 3)
y = y.to(device)
print("y.device:", y.device)

# Operations between x and y work because they're on the same device
z = x + y
print("z.device:", z.device)

---
## Summary

| Task | Code |
|------|------|
| From list | `torch.tensor([1, 2, 3])` |
| Zeros / ones | `torch.zeros(m, n)` / `torch.ones(m, n)` |
| Random uniform | `torch.rand(m, n)` |
| Random normal | `torch.randn(m, n)` |
| Range | `torch.arange(start, stop, step)` |
| Evenly spaced | `torch.linspace(start, stop, n)` |
| Shape | `x.shape` |
| Reshape | `x.reshape(m, n)` or `x.view(m, n)` |
| Move to GPU | `x.to('cuda')` |
| To NumPy | `x.numpy()` |

**Next:** [02_broadcasting.ipynb](02_broadcasting.ipynb). We next discuss how PyTorch handles tensors of different shapes during operations.